# GPT-4o Comparison — Three Baselines

**Thesis Section: Chapter 4.4**

Runs GPT-4o as a comparison model under three conditions:

| Baseline | Method |
|----------|--------|
| Naive    | Raw time series, open-ended question, no MCQ |
| MCQ-only | MCQ template, middle 512 pts, no stats, no pre-screen |
| Hybrid   | Same pre-screen + stats + MCQ as ChatTS Approach 3 |

The hybrid condition tests whether GPT-4o benefits from the same prompt engineering
as ChatTS, enabling a fair head-to-head comparison.

**Requires**: `OPENAI_API_KEY` in `.env`

In [ ]:
import sys
sys.path.insert(0, '../..')

from dotenv import load_dotenv
load_dotenv('../../.env')

import os
import numpy as np

from src.data.timeseer_client import fetch_series_api, list_series_api, AREAS
from src.data.ground_truth import GROUND_TRUTH
from src.prescreener.analyze import hybrid_analyze
from src.prompts.templates import MCQ_CATEGORIES
from src.inference.openai_infer import ask_openai_chunk, ask_openai_naive, ask_openai_mcq_only
from src.parsing.extract import extract_category
from src.evaluation.metrics import compute_metrics
from src.evaluation.report import print_summary_table, save_results

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env'
print('Imports OK')

In [ ]:
AREA = 'Reactor 1'
tags = list_series_api(AREA)
print(f'{len(tags)} signals in {AREA}')

In [ ]:
# ── Baseline 1: Naive GPT-4o ─────────────────────────────────────
print('Running NAIVE baseline (no MCQ, no stats, no pre-screen)...')
results_naive = []

for tag in tags:
    vals, idx = fetch_series_api(tag, AREA)
    if vals is None: continue

    mid   = len(vals) // 2
    chunk = [vals[max(0, mid-256):mid+256]]
    answer = ask_openai_naive(chunk, tag)

    # Simple keyword extraction for naive baseline
    up = answer.upper()
    if any(w in up for w in ['NO ANOMALY', 'LOOKS CLEAN', 'NORMAL', 'NO SIGNIFICANT']):
        cat, lbl = 'E', 'None/Clean'
    elif 'SPIKE' in up or 'OUTLIER' in up:
        cat, lbl = 'B', 'Spikes'
    elif 'DRIFT' in up or 'GRADUAL' in up:
        cat, lbl = 'A', 'Drift'
    elif 'FLAT' in up or 'FROZEN' in up or 'CONSTANT' in up:
        cat, lbl = 'C', 'Frozen'
    elif 'NEGATIVE' in up or 'IMPOSSIBLE' in up:
        cat, lbl = 'L', 'Intermittent'
    else:
        cat, lbl = '?', 'Unclear'

    gt = GROUND_TRUTH.get(tag, '?')
    results_naive.append({'Tag': tag, 'gt': gt, 'Category': cat, 'Label': lbl})
    print(f'  {tag:<30} → {cat}) {lbl}  (GT: {gt})')

In [ ]:
# ── Baseline 2: MCQ-only GPT-4o ──────────────────────────────────
print('Running MCQ-ONLY baseline...')
results_mcq = []

for tag in tags:
    vals, idx = fetch_series_api(tag, AREA)
    if vals is None: continue

    answer = ask_openai_mcq_only(vals, tag, MCQ_CATEGORIES)
    cat, lbl = extract_category(answer)
    gt = GROUND_TRUTH.get(tag, '?')
    results_mcq.append({'Tag': tag, 'gt': gt, 'Category': cat, 'Label': lbl})
    print(f'  {tag:<30} → {cat}) {lbl}  (GT: {gt})')

In [ ]:
# ── Baseline 3: Hybrid GPT-4o (same prompt as ChatTS Approach 3) ──
print('Running HYBRID baseline (same pre-screen + MCQ as ChatTS)...')
results_hybrid = []

for tag in tags:
    vals, idx = fetch_series_api(tag, AREA)
    if vals is None: continue

    chunk, question, tname, detected, chunk_desc = hybrid_analyze(vals, idx, tag)
    answer = ask_openai_chunk(chunk, question)
    cat, lbl = extract_category(answer, detected)

    gt = GROUND_TRUTH.get(tag, '?')
    results_hybrid.append({'Tag': tag, 'gt': gt, 'Detected': ', '.join(detected),
                           'Category': cat, 'Label': lbl})
    print(f'  {tag:<30} → {cat}) {lbl}  (GT: {gt})')

In [ ]:
# ── Compare all three ─────────────────────────────────────────────
def acc(results):
    n = [r for r in results if r['gt'] != '?']
    if not n: return 0
    correct = sum(r['Category'] == r['gt'] for r in n)
    return correct / len(n)

print('\n' + '='*55)
print(f'GPT-4o COMPARISON — {AREA}')
print('='*55)
print(f'  Naive    : {acc(results_naive)*100:.1f}%')
print(f'  MCQ-only : {acc(results_mcq)*100:.1f}%')
print(f'  Hybrid   : {acc(results_hybrid)*100:.1f}%')
print('='*55)

save_results(results_hybrid, f'../../data/gpt4o_hybrid_{AREA.replace(" ","_")}.txt',
             header=f'GPT-4o Hybrid | {AREA}')